In [ ]:
import numpy as np

In [ ]:
import pandas as pd

df = pd.read_csv('/content/Coursera.csv')


In [ ]:
df.head()

,partner,course,skills,review
0,Google,Google Cybersecurity,"{"" Network Security"","" Python Programming"","" L...","[Very Good, The course does not award academic..."
1,Google,Google Data Analytics,"{"" Data Analysis"","" R Programming"","" SQL"","" Bu...","[Very Good, Completing this course counts for ..."
2,Google,Google Project Management:,"{"" Project Management"","" Strategy and Operatio...","[Very Good, Completing this course counts for ..."
3,Google,Google Digital Marketing & E-commerce,"{"" Digital Marketing"","" Marketing"","" Marketing...","[Very Good, The course does not award academic..."
4,Google,Google IT Support,"{"" Computer Networking"","" Network Architecture...","[Very Good, Completing this course counts for ..."


In [ ]:
def rate(rating):
  l=[]
  if rating>4.8:
    l.append("Excellent")
  elif rating>4.5:
    l.append('Very Good')
  elif rating>4.0:
    l.append('Good')
  else:
    l.append('basic')
  return l
df['review'] = df['rating'].apply(rate)

In [ ]:
df["review"] = df.apply(
    lambda row: row["review"] +
        ["Completing this course counts for academic credit"]
        if row["crediteligibility"]
        else row["review"] +
        ["The course does not award academic credit"],
    axis=1
)

In [ ]:
df['review'] = df.apply(
    lambda row: list(row["review"]) + [row["level"],row['certificatetype'],row['duration']],
    axis=1
)

In [ ]:
df = df.drop(['level', 'duration','crediteligibility','reviewcount','rating'],axis=1)

In [ ]:
df = df.drop('certificatetype',axis=1)

In [ ]:
import re
def clean_item(item):
    if not isinstance(item, str):
        return ""
    item = item.strip()
    item = re.sub(r"\s+", " ", item)
    return item.lower()

In [ ]:
def merge_review_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = [clean_item(x) for x in lst]
    return " | ".join(cleaned)

In [ ]:
df['review'] = df['review'].apply(merge_review_list)

In [ ]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
sentiment_model = pipeline("sentiment-analysis")

def get_sentiment_score(text):
    try:
        out = sentiment_model(text[:512])[0]
        return 1 if out["label"] == "POSITIVE" else 0
    except:
        return 0

df["sentiment_score"] = df["review"].apply(get_sentiment_score)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


In [ ]:
sbert = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = sbert.encode(
    df["review"].astype(str).tolist(),
    convert_to_numpy=True
)

In [ ]:
df["embedding"] = list(embeddings)

In [ ]:
df.head()

,partner,course,skills,review,sentiment_score,embedding
0,Google,Google Cybersecurity,"{"" Network Security"","" Python Programming"","" L...",very good | the course does not award academic...,1,"[-0.0070385397, -0.027633997, -0.0072604744, 0..."
1,Google,Google Data Analytics,"{"" Data Analysis"","" R Programming"","" SQL"","" Bu...",very good | completing this course counts for ...,1,"[-0.00056329585, -0.042549968, -0.005556102, 0..."
2,Google,Google Project Management:,"{"" Project Management"","" Strategy and Operatio...",very good | completing this course counts for ...,1,"[-0.00056329585, -0.042549968, -0.005556102, 0..."
3,Google,Google Digital Marketing & E-commerce,"{"" Digital Marketing"","" Marketing"","" Marketing...",very good | the course does not award academic...,1,"[-0.0070385397, -0.027633997, -0.0072604744, 0..."
4,Google,Google IT Support,"{"" Computer Networking"","" Network Architecture...",very good | completing this course counts for ...,1,"[-0.00056329585, -0.042549968, -0.005556102, 0..."


In [ ]:
def recommend(course_index, top_k=10, alpha=0.6):
    # matrix of all embeddings
    emb_matrix = np.vstack(df["embedding"].values)

    # target emb
    target = df.loc[course_index, "embedding"].reshape(1, -1)

    # cosine similarity
    sim = cosine_similarity(target, emb_matrix)[0]

    # normalize similarity
    sim_norm = (sim - sim.min()) / (sim.max() - sim.min() + 1e-9)

    # normalize sentiment
    sent = df["sentiment_score"].values.astype(float)
    sent_norm = (sent - sent.min()) / (sent.max() - sent.min() + 1e-9)

    # hybrid final score
    final = alpha * sim_norm + (1 - alpha) * sent_norm

    df["final_score"] = final

    return df.sort_values("final_score", ascending=False)[
        ["review", "final_score"]
    ].head(top_k)

In [ ]:
recommend(0, top_k=5)

,review,final_score
3,very good | the course does not award academic...,1.0
0,very good | the course does not award academic...,1.0
11,very good | the course does not award academic...,1.0
22,very good | the course does not award academic...,1.0
518,very good | the course does not award academic...,1.0
